# Assignment: Barcelona

## Prep

### Imports, shared definitions, datasets

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily
import pointpats
import numpy as np
import seaborn as sns
from libpysal import graph
import esda

In [ ]:
def listings(city_path, quarter_end_dates):
    """load listings for a city_path for a series of dates (representing quarter ends) and combines into one DataFrame.

    Adds a new `quarter_end_date` column so that data for each quarter can still be pulled out later.
    """
    listings_url_base = f"https://data.insideairbnb.com/{city_path}"
    dfs = []
    for quarter_end_date in quarter_end_dates:
        listings_url = f"{listings_url_base}/{quarter_end_date}/data/listings.csv.gz"
        df = pd.read_csv(listings_url, compression='gzip')
        df['quarter_end_date'] = quarter_end_date
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [ ]:
listings_df = listings("spain/catalonia/barcelona", ["2024-12-12","2025-03-05","2025-06-12","2025-09-14"])
listings_df

In [ ]:
def create_gdf_from_latlon(df):
    """creates a new GDF from a Dataframe containing `latitude` and `longitude` columns."""
    geometry = gpd.points_from_xy(df['longitude'], df['latitude'], crs="EPSG:4326")
    return gpd.GeoDataFrame(df, geometry=geometry)

In [ ]:
listings_gdf = create_gdf_from_latlon(listings_df)
listings_gdf

In [ ]:
cols = listings_gdf.columns
type_cols = cols[cols.str.contains("type")]
type_cols

In [ ]:
def types_only(gdf):
    """Takes a GDF and finds all columns whose name contains `type`, and filters out any rows with no values or empty string.

    It keeps any columns that related to identity.
    """
    types_gdf = gdf.copy(deep=True)
    
    # find columns related to `type`
    columns = types_gdf.columns
    type_cols = list(cols[cols.str.contains("type")])
    keep_cols = type_cols + ["id",types_gdf.geometry.name]
    types_gdf = types_gdf[keep_cols]

    # remove any rows with empty strings or missing values
    for col in type_cols:
        types_gdf[col] = (
            types_gdf[col].str.replace(r"^\s*$", "", regex=True).replace("", None)
        )
    types_gdf = types_gdf.dropna()
    
    return types_gdf
    
listings_types_gdf = types_only(listings_gdf)
listings_types_gdf

In [ ]:
listings_types_gdf.explore("property_type", tiles="CartoDB Positron", prefer_canvas=True)

In [ ]:
listings_types_gdf.explore("room_type", tiles="CartoDB Positron", prefer_canvas=True)

In [ ]:
from libpysal import graph

In [ ]:
knn5 = graph.Graph.build_knn(listings_types_gdf, k=5)
knn5

In [ ]:
# how many rows per location?

In [ ]:
sns.displot(listings_types_gdf.groupby("id")["id"].count())

In [ ]:
sns.displot(listings_types_gdf.groupby("geometry")["geometry"].count())

In [ ]:
sns.displot(listings_types_gdf.groupby("geometry")["id"].count())

In [ ]:
sns.displot(listings_types_gdf.groupby("geometry")["id"].max())

In [ ]:
listings_types_gdf.groupby("geometry")["id"].max()

In [ ]:
unique_by_geometry_gbf = listings_types_gdf.groupby("geometry")[["id"]].max()
unique_by_geometry_gbf


In [ ]:
reduced_gdf = listings_types_gdf.merge(unique_by_geometry_gbf, on="id", how="right").dropna()
reduced_gdf

In [ ]:
pd.read_csv("BarcelonaCiutat_SeccionsCensals.csv")

In [ ]:
from shapely import from_wkt

def barcelona_census_areas():
    url = "BarcelonaCiutat_SeccionsCensals.csv"
    df = pd.read_csv(url)
    
    df["geometry"] = df["geometria_wgs84"].apply(from_wkt)
    df = df.drop(["geometria_etrs89","geometria_wgs84"],axis=1)
    return gpd.GeoDataFrame(df,geometry="geometry",crs="EPSG:4326")

In [ ]:
census_areas_gdf = barcelona_census_areas()
census_areas_gdf.explore()

In [ ]:
import tobler

grid_gdf = tobler.util.h3fy(census_areas_gdf, resolution=9)

In [ ]:
grid_gdf.explore()

In [ ]:
grid_gdf